In [7]:
#%pip install -U sentence-transformers scikit-learn

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import json 

from pathlib import Path 
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
PROCESSED_PATH = Path("../data/processed")
EMBEDDINGS_PATH = Path("../data/embeddings")

EMBEDDINGS_PATH.mkdir(parents=True, exist_ok=True)

## Wczytanie datasetu przygotowanego w 02_Preprocessing.ipynb

In [10]:
df = pd.read_parquet(PROCESSED_PATH / "movies_processed.parquet")

print(df.shape)
display(df.head())

(45430, 20)


,id,title,overview,genres,runtime,vote_average,vote_count,release_date,original_language,popularity,imdb_id,release_year,genres_list,keywords_list,directors,top_cast,keywords_text,genres_text,embedding_text,runtime_category
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",81.0,7.7,5415.0,1995-10-30,en,21.946943,tt0114709,1995.0,"[Animation, Comedy, Family]","[jealousy, toy, boy, friendship, friends, riva...",[John Lasseter],"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...","jealousy, toy, boy, friendship, friends, rival...","Animation, Comedy, Family","Plot: Led by Woody, Andy's toys live happily i...",short
1,8844,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",104.0,6.9,2413.0,1995-12-15,en,17.015539,tt0113497,1995.0,"[Adventure, Fantasy, Family]","[board game, disappearance, based on children'...",[Joe Johnston],"[Robin Williams, Jonathan Hyde, Kirsten Dunst,...","board game, disappearance, based on children's...","Adventure, Fantasy, Family",Plot: When siblings Judy and Peter discover an...,medium
2,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",101.0,6.5,92.0,1995-12-22,en,11.712900,tt0113228,1995.0,"[Romance, Comedy]","[fishing, best friend, duringcreditsstinger, o...",[Howard Deutch],"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...","fishing, best friend, duringcreditsstinger, ol...","Romance, Comedy",Plot: A family wedding reignites the ancient f...,medium
3,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",127.0,6.1,34.0,1995-12-22,en,3.859495,tt0114885,1995.0,"[Comedy, Drama, Romance]","[based on novel, interracial relationship, sin...",[Forest Whitaker],"[Whitney Houston, Angela Bassett, Loretta Devi...","based on novel, interracial relationship, sing...","Comedy, Drama, Romance","Plot: Cheated on, mistreated and stepped on, t...",long
4,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]",106.0,5.7,173.0,1995-02-10,en,8.387519,tt0113041,1995.0,[Comedy],"[baby, midlife crisis, confidence, aging, daug...",[Charles Shyer],"[Steve Martin, Diane Keaton, Martin Short, Kim...","baby, midlife crisis, confidence, aging, daugh...",Comedy,Plot: Just when George Banks has recovered fro...,medium


In [11]:
display(df[
    [
        "id", "title", "embedding_text", "genres_list"
    ]
].head())

,id,title,embedding_text,genres_list
0,862,Toy Story,"Plot: Led by Woody, Andy's toys live happily i...","[Animation, Comedy, Family]"
1,8844,Jumanji,Plot: When siblings Judy and Peter discover an...,"[Adventure, Fantasy, Family]"
2,15602,Grumpier Old Men,Plot: A family wedding reignites the ancient f...,"[Romance, Comedy]"
3,31357,Waiting to Exhale,"Plot: Cheated on, mistreated and stepped on, t...","[Comedy, Drama, Romance]"
4,11862,Father of the Bride Part II,Plot: Just when George Banks has recovered fro...,[Comedy]


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

print("Embeddings dimensions: ", model.get_sentence_embedding_dimension())
print("Max sequence length: ", model.max_seq_length)

Device: cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings dimensions:  384
Max sequence length:  256


C:\Users\Patrycja\AppData\Local\Temp\ipykernel_15836\3881499816.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embeddings dimensions: ", model.get_sentence_embedding_dimension())


## Przygotowanie tekstów

In [13]:
df["embedding_text"] = (
    df["embedding_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

texts = df["embedding_text"].tolist()

print("Number of movies:", len(texts))
print("Empty texts:", sum(not text for text in texts))

Number of movies: 45430
Empty texts: 754


In [14]:
token_lengths = [
    len(tokens)
    for tokens in model.tokenizer(
        texts,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]
]

print("Maximum tokens:", max(token_lengths))
print("Median tokens:", np.median(token_lengths))

print("Texts exceeding model limit:",
    sum(length > model.max_seq_length for length in token_lengths)
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (273 > 256). Running this sequence through the model will result in indexing errors


Maximum tokens: 651
Median tokens: 76.0
Texts exceeding model limit: 65


## Generowanie embeddingów

In [15]:
valid_mask = df["embedding_text"].str.len() > 0
valid_indices = np.flatnonzero(valid_mask.to_numpy())

valid_texts = df.loc[valid_mask, "embedding_text"].tolist()

print("Movies with text:", len(valid_texts))

Movies with text: 44676


In [16]:
text_embeddings_valid = model.encode(
    valid_texts,
    batch_size=64 if device == "cuda" else 32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(text_embeddings_valid.shape)

Batches:   0%|          | 0/1397 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
embedding_dim = model.get_sentence_embedding_dimension()

text_embeddings = np.zeros(
    (len(df), embedding_dim),
    dtype=np.float32
)

text_embeddings[valid_indices] = text_embeddings_valid

In [ ]:
print("Shape:", text_embeddings.shape)
print("Data type:", text_embeddings.dtype)
print("NaN values:", np.isnan(text_embeddings).sum())

### Wizualizacja norm wektorów

In [ ]:
norms = np.linalg.norm(text_embeddings, axis=1)

plt.figure(figsize=(10, 5))
plt.hist(norms, bins=30)

plt.title("Distribution of embedding norms")
plt.xlabel("L2 norm")
plt.ylabel("Number of movies")

plt.show()

## Multi-Hot Encoding gatunków

In [ ]:
mlb = MultiLabelBinarizer()

genres_embeddings = mlb.fit_transform(df["genres_list"].apply(
    lambda x: x if isinstance(x, list) else []
)).astype(np.float32)

print("Genres shape:", genres_embeddings.shape)
print("Genres:", mlb.classes_)

## Test podobieństwa filmów

In [ ]:
def find_similar_movies(title, top_n=10):
    matches = df[
        df["title"].str.lower() == title.lower()
    ]

    if matches.empty:
        print("Movie not found")
        return

    idx = matches.index[0]

    if not valid_mask.iloc[idx]:
        print("This movie has no text embedding")
        return

    similarity = text_embeddings @ text_embeddings[idx]

    similarity[~valid_mask.to_numpy()] = -np.inf
    similarity[idx] = -np.inf

    similar_indices = np.argsort(similarity)[::-1][:top_n]

    results = df.iloc[similar_indices][
        ["title", "genres_list", "vote_average"]
    ].copy()

    results["similarity"] = similarity[similar_indices]

    return results

In [ ]:
df = df.reset_index(drop=True)
display(find_similar_movies("Interstellar"))

In [ ]:
np.save(EMBEDDINGS_PATH / "text_embeddings.npy", text_embeddings)
np.save(EMBEDDINGS_PATH / "genres_embeddings.npy", genres_embeddings)
np.save(EMBEDDINGS_PATH / "movie_ids.npy", df["id"].to_numpy())

### Zapisywanie nazwy gatunków i konfigurację

In [ ]:
config = {
    "model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dimension": embedding_dim,
    "normalized": True,
    "genre_names": mlb.classes_.tolist(),
    "number_of_movies": len(df)
}

with open(EMBEDDINGS_PATH / "embedding_config.json", "w", encoding="utf-8") as file:
    json.dump(config, file, indent=4)

In [ ]:
loaded_text = np.load(
    EMBEDDINGS_PATH / "text_embeddings.npy"
)

loaded_genres = np.load(
    EMBEDDINGS_PATH / "genres_embeddings.npy"
)

loaded_ids = np.load(
    EMBEDDINGS_PATH / "movie_ids.npy"
)

assert len(loaded_text) == len(loaded_genres) == len(loaded_ids)
assert np.array_equal(loaded_ids, df["id"].to_numpy())
assert np.isfinite(loaded_text).all()
assert np.isfinite(loaded_genres).all()

print("Embeddings saved successfully!")